Example 14-2. Setting up a logger in Python

In [ ]:
import csv
import io
import logging
from datetime import datetime

logger_name = "processing_bronze_to_silver"


class CsvFormatter(logging.Formatter):
    def format(self, record):
        output = io.StringIO()
        csv.writer(output, lineterminator="").writerow(
            [
                self.formatTime(record, "%Y-%m-%d %H:%M:%S"),
                record.levelname,
                record.name,
                record.getMessage(),
            ]
        )
        return output.getvalue()


logger = logging.getLogger(logger_name)
logger.setLevel(logging.DEBUG)

stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.setLevel(logging.INFO)

filename = datetime.now().strftime(f"{logger_name}_%Y%m%d_%H%M%S.csv")
log_file_path = f"/lakehouse/default/Files/{filename}"
file_handler = logging.FileHandler(log_file_path, encoding="utf-8")
file_handler.setLevel(logging.DEBUG)
file_handler.setFormatter(CsvFormatter())

logger.addHandler(stream_handler)
logger.addHandler(file_handler)

logger.debug("Is only written to the CSV file")
logger.info("Is written to cell output and the CSV file")
logger.warning("Is written to cell output and the CSV file")
logger.error("Is written to cell output and the CSV file")
logger.fatal("Is written to cell output and the CSV file")

logging.shutdown()

Example 14-3. How to read CSV log files with Python and PySpark

In [ ]:
import pyarrow.csv as csv
import pyarrow.dataset as ds

# Using Apache Arrow to read log files
df = ds.dataset(
    "/lakehouse/default/Files/logs",
    format=ds.CsvFileFormat(
        read_options=csv.ReadOptions(autogenerate_column_names=True)
    ),
)
# displaying the logfile as table
display(df)
# Using Spark to read log files
df = spark.read.format("csv").option("header", "false").load("Files/logs")
# displaying the log file as table
display(df)

Example 14-6. Measuring the quality of a table

In [ ]:
import great_expectations as gx
import great_expectations.expectations as gxe

# Create DataFrame from customers table
df = spark.table("customers")
# Creating a Great Expectations context
# ephemeral is in memory
context = gx.get_context(mode="ephemeral")
# Create definitions of data_sources, DataFrame and a
# name to the batch containing the data being measured.
# Names are for references in Great Expectations and are
# not correlated with lakehouses, tables, or spark
data_source = context.data_sources.add_spark(name="silver")
data_asset = data_source.add_dataframe_asset(name="customers")
batch_definition = data_asset.add_batch_definition_whole_dataframe("customers_batch")
# Adding the DataFrame being measured to the batch
# definition
batch = batch_definition.get_batch(batch_parameters={"dataframe": df})
# Add a test for nulls to the id column of the dataset
# and add a test for the column with unique values
customers_suite = gx.ExpectationSuite(
    name="customers_quality",
    expectations=[
        gxe.ExpectColumnValuesToNotBeNull(column="id"),
        gxe.ExpectColumnValuesToBeUnique(column="id"),
    ],
)
# Measure quality by running the tests
result = batch.validate(customers_suite)
# print a report
print("Success:", result.success)
print(result)